# 🎵 Quarry Studio — Own Music Batch Generator (ACE-Step, Apache 2.0)

Generates **40+ mood-tagged instrumental tracks** that are **100% yours** (Apache 2.0 — commercial use, no attribution, claim-proof for YouTube).

**Runtime menu → Change runtime type → T4 GPU** before running.

Flow: run cells top to bottom (~60–90 min total) → cell 5 downloads `own-music-library.zip` → optional cell 6 auto-uploads it to GitHub.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 — Install ACE-Step (~4 min)

In [ ]:
%cd /content
!rm -rf ACE-Step
!git clone https://github.com/ace-step/ACE-Step.git
%cd ACE-Step
!pip install -e . -q
!pip install pydub -q
print('✅ ACE-Step installed')

## 2 — The batch plan (48 tracks across 6 moods)

Each entry: (mood folder, seed number, style tags, duration seconds). Edit freely.


In [ ]:
MOOD_PLAN = [
  ('calm', 1, 'ambient cinematic calm, soft piano, warm pads, slow, peaceful, instrumental', 120),
  ('calm', 2, 'ambient cinematic calm, soft piano, warm pads, slow, peaceful, instrumental', 120),
  ('calm', 3, 'ambient cinematic calm, soft piano, warm pads, slow, peaceful, instrumental', 120),
  ('calm', 4, 'ambient cinematic calm, soft piano, warm pads, slow, peaceful, instrumental', 120),
  ('calm', 5, 'ambient cinematic calm, soft piano, warm pads, slow, peaceful, instrumental', 120),
  ('calm', 6, 'ambient cinematic calm, soft piano, warm pads, slow, peaceful, instrumental', 120),
  ('calm', 101, 'gentle neoclassical strings and piano, contemplative, soft, instrumental', 120),
  ('calm', 102, 'gentle neoclassical strings and piano, contemplative, soft, instrumental', 120),
  ('calm', 103, 'gentle neoclassical strings and piano, contemplative, soft, instrumental', 120),
  ('calm', 104, 'gentle neoclassical strings and piano, contemplative, soft, instrumental', 120),
  ('calm', 105, 'gentle neoclassical strings and piano, contemplative, soft, instrumental', 120),
  ('calm', 106, 'gentle neoclassical strings and piano, contemplative, soft, instrumental', 120),
  ('epic', 1, 'epic cinematic orchestral, heroic drums, rising tension, powerful, instrumental', 90),
  ('epic', 2, 'epic cinematic orchestral, heroic drums, rising tension, powerful, instrumental', 90),
  ('epic', 3, 'epic cinematic orchestral, heroic drums, rising tension, powerful, instrumental', 90),
  ('epic', 4, 'epic cinematic orchestral, heroic drums, rising tension, powerful, instrumental', 90),
  ('epic', 5, 'epic cinematic orchestral, heroic drums, rising tension, powerful, instrumental', 90),
  ('epic', 101, 'dark epic trailer music, braams, percussion, motivational, instrumental', 90),
  ('epic', 102, 'dark epic trailer music, braams, percussion, motivational, instrumental', 90),
  ('epic', 103, 'dark epic trailer music, braams, percussion, motivational, instrumental', 90),
  ('epic', 104, 'dark epic trailer music, braams, percussion, motivational, instrumental', 90),
  ('epic', 105, 'dark epic trailer music, braams, percussion, motivational, instrumental', 90),
  ('driving', 1, 'driving electronic beat, pulsing synth bass, focused energy, modern, instrumental', 90),
  ('driving', 2, 'driving electronic beat, pulsing synth bass, focused energy, modern, instrumental', 90),
  ('driving', 3, 'driving electronic beat, pulsing synth bass, focused energy, modern, instrumental', 90),
  ('driving', 4, 'driving electronic beat, pulsing synth bass, focused energy, modern, instrumental', 90),
  ('driving', 101, 'corporate driving rhythm, clean plucks, steady momentum, instrumental', 90),
  ('driving', 102, 'corporate driving rhythm, clean plucks, steady momentum, instrumental', 90),
  ('driving', 103, 'corporate driving rhythm, clean plucks, steady momentum, instrumental', 90),
  ('driving', 104, 'corporate driving rhythm, clean plucks, steady momentum, instrumental', 90),
  ('bright', 1, 'bright acoustic pop, uplifting light percussion, optimistic, instrumental', 90),
  ('bright', 2, 'bright acoustic pop, uplifting light percussion, optimistic, instrumental', 90),
  ('bright', 3, 'bright acoustic pop, uplifting light percussion, optimistic, instrumental', 90),
  ('bright', 4, 'bright acoustic pop, uplifting light percussion, optimistic, instrumental', 90),
  ('bright', 101, 'sunny ukulele and claps, feel-good, light, instrumental', 90),
  ('bright', 102, 'sunny ukulele and claps, feel-good, light, instrumental', 90),
  ('bright', 103, 'sunny ukulele and claps, feel-good, light, instrumental', 90),
  ('bright', 104, 'sunny ukulele and claps, feel-good, light, instrumental', 90),
  ('luxury', 1, 'elegant neoclassical strings and grand piano, sophisticated, slow waltz, instrumental', 120),
  ('luxury', 2, 'elegant neoclassical strings and grand piano, sophisticated, slow waltz, instrumental', 120),
  ('luxury', 3, 'elegant neoclassical strings and grand piano, sophisticated, slow waltz, instrumental', 120),
  ('luxury', 101, 'quiet luxury jazz lounge, smooth brushes, warm saxophone, classy, instrumental', 120),
  ('luxury', 102, 'quiet luxury jazz lounge, smooth brushes, warm saxophone, classy, instrumental', 120),
  ('luxury', 103, 'quiet luxury jazz lounge, smooth brushes, warm saxophone, classy, instrumental', 120),
  ('dark', 1, 'dark cinematic tension, subtle drones, mystery, documentary, instrumental', 120),
  ('dark', 2, 'dark cinematic tension, subtle drones, mystery, documentary, instrumental', 120),
  ('dark', 3, 'dark cinematic tension, subtle drones, mystery, documentary, instrumental', 120),
  ('dark', 4, 'dark cinematic tension, subtle drones, mystery, documentary, instrumental', 120),
]
print(f'{len(MOOD_PLAN)} tracks planned')

## 3 — Generate (~60–90 min total; progress prints per track)

In [ ]:
import os, subprocess, time
os.makedirs('/content/own-music', exist_ok=True)

from acestep.pipeline import ACEStepPipeline
pipe = ACEStepPipeline(checkpoint_dir='/content/checkpoints')

done, failed = 0, []
t0 = time.time()
for mood, seed, tags, dur in MOOD_PLAN:
    out_dir = f'/content/own-music/{mood}'
    os.makedirs(out_dir, exist_ok=True)
    out_name = f'qs-{mood}-{seed:03d}'
    if os.path.exists(f'{out_dir}/{out_name}.mp3'):
        done += 1
        continue
    try:
        res = pipe(
            prompt=f'{tags}, professional background music',
            tags=f'{mood}, instrumental',
            lyrics='[inst]',
            infer_steps=60,
            guidance_scale=15.0,
            duration=float(dur),
            seed=seed,
        )
        wav = res[0] if isinstance(res, (list, tuple)) else res
        if isinstance(wav, dict):
            wav = wav.get('audio') or wav.get('filepath')
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(wav),
                        '-af', 'loudnorm=I=-16:TP=-1.5:LRA=11',
                        '-codec:a', 'libmp3lame', '-b:a', '192k',
                        f'{out_dir}/{out_name}.mp3'], check=True)
        done += 1
        print(f'✅ [{done}/{len(MOOD_PLAN)}] {out_name}.mp3 ({round(time.time()-t0)}s elapsed)')
    except Exception as e:
        failed.append((out_name, str(e)[:80]))
        print(f'❌ {out_name}: {str(e)[:80]}')
    time.sleep(2)

print(f'DONE: {done} tracks, {len(failed)} failed')
if failed:
    print('failed:', failed)

## 4 — Build the manifest + zip

In [ ]:
import json, glob, os
tracks = []
for path in sorted(glob.glob('/content/own-music/*/*.mp3')):
    mood = path.split('/')[-2]
    name = os.path.basename(path)
    num = name.replace('.mp3', '').split('-')[-1].upper()
    tracks.append({'file': f'{mood}/{name}', 'mood': mood,
                   'title': f'Quarry {mood.capitalize()} {num}', 'tags': mood})
manifest = {'version': 1,
            'library': 'Quarry Studio Originals — self-produced with ACE-Step (Apache 2.0), no attribution required',
            'tracks': tracks}
json.dump(manifest, open('/content/own-music/manifest.json', 'w'), indent=2)
print(f'manifest: {len(tracks)} tracks')
!cd /content && zip -rq own-music-library.zip own-music
!ls -lh /content/own-music-library.zip

## 5 — Download the zip
Run this cell, then grab `own-music-library.zip` from the left file panel.

In [ ]:
from google.colab import files
files.download('/content/own-music-library.zip')

## 6 (optional, easiest) — Auto-upload to GitHub
Paste a GitHub Personal Access Token (repo scope). Creates release `own-music-v1` on `muzzamilhassan/automation` with the manifest + all mp3s. **The music engine picks it up automatically — no other step needed.**

In [ ]:
GH_TOKEN = ''  # @param {type:'string'}
import json, glob, os, urllib.request

repo = 'muzzamilhassan/automation'
tag = 'own-music-v1'
def hdr(): return {'Authorization': f'Bearer {GH_TOKEN}', 'Content-Type': 'application/json', 'User-Agent': 'colab-music'}
def api(url, method='GET', body=None):
    req = urllib.request.Request(url, method=method, headers=hdr(),
                                 data=json.dumps(body).encode() if body else None)
    return urllib.request.urlopen(req)

try:
    rel = json.load(api(f'https://api.github.com/repos/{repo}/releases/tags/{tag}'))
except Exception:
    rel = json.load(api(f'https://api.github.com/repos/{repo}/releases', 'POST',
                        {'tag_name': tag, 'name': 'Quarry Originals — own music library',
                         'body': 'Self-produced ACE-Step instrumentals (Apache 2.0). Claim-proof.'}))
print('release ready:', rel['tag_name'])

existing = {a['name'] for a in rel.get('assets', [])}
files = ['/content/own-music/manifest.json'] + sorted(glob.glob('/content/own-music/*/*.mp3'))
for f in files:
    name = os.path.basename(f) if 'manifest' in f else '/'.join(f.split('/')[-2:])
    if name in existing:
        print('skip (exists):', name)
        continue
    data = open(f, 'rb').read()
    req = urllib.request.Request(f"https://uploads.github.com/repos/{repo}/releases/{rel['id']}/assets?name={name}",
                                 method='POST', data=data,
                                 headers={'Authorization': f'Bearer {GH_TOKEN}', 'Content-Type': 'application/octet-stream', 'User-Agent': 'colab-music'})
    urllib.request.urlopen(req)
    print('uploaded:', name)
print('✅ all uploaded — the music engine picks this up automatically')